In [ ]:
from pathlib import Path
import os
import json
import random
import shutil
import gzip
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"
CLP_OUT_DIR = OUT_DIR / "clp"

DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)
CLP_OUT_DIR.mkdir(exist_ok=True)

RAW_PATH = DATA_DIR / "synthetic_logs.jsonl"

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)

ROOT: /content
DATA_DIR: /content/data
OUT_DIR: /content/outputs


In [ ]:
random.seed(0)

services = ["auth", "payment", "search", "feed", "profile"]
levels = ["INFO", "WARN", "ERROR"]
paths = ["/login", "/logout", "/checkout", "/search", "/profile", "/feed"]
methods = ["GET", "POST"]
regions = ["us-east", "us-west", "eu-central"]

def make_record(i: int) -> dict:
    service = random.choice(services)
    level = random.choices(levels, weights=[0.80, 0.15, 0.05])[0]

    record = {
        "timestamp": f"2024-01-01T00:{(i // 60) % 60:02d}:{i % 60:02d}.000Z",
        "level": level,
        "service": service,
        "host": f"{service}-{random.randint(1, 20)}",
        "trace_id": f"trace-{random.randint(1, 5000)}",
        "request": {
            "method": random.choice(methods),
            "path": random.choice(paths),
            "region": random.choice(regions),
        },
        "latency_ms": random.randint(1, 500),
        "message": random.choice([
            "request completed",
            "request retried",
            "dependency timeout",
            "cache miss",
            "cache hit",
            "invalid user token",
        ]),
    }

    # Controlled schema variation by service.
    if service == "payment":
        record["payment"] = {
            "currency": random.choice(["USD", "EUR", "GBP"]),
            "amount_bucket": random.choice(["small", "medium", "large"]),
            "processor": random.choice(["stripe", "adyen", "internal"]),
        }
    elif service == "search":
        record["search"] = {
            "query_type": random.choice(["keyword", "semantic", "autocomplete"]),
            "num_results": random.randint(0, 100),
        }
    elif service == "auth":
        record["auth"] = {
            "provider": random.choice(["password", "google", "github"]),
            "success": level != "ERROR",
        }

    return record

def write_dataset(n_records: int, path: Path) -> None:
    with path.open("w") as f:
        for i in range(n_records):
            f.write(json.dumps(make_record(i), separators=(",", ":")) + "\n")

N = 100_000
write_dataset(N, RAW_PATH)

raw_size = RAW_PATH.stat().st_size
print(f"records = {N:,}")
print(f"raw size = {raw_size / 1e6:.2f} MB")

records = 100,000
raw size = 25.99 MB


In [ ]:
def file_size(path: Path) -> int:
    return Path(path).stat().st_size

def dir_size(path: Path) -> int:
    path = Path(path)
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

def run(cmd, *, cwd=None, check=True):
    print("+", " ".join(map(str, cmd)))
    out = subprocess.run(
        list(map(str, cmd)),
        cwd=cwd,
        check=check,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(out.stdout[-2000:])
    return out

In [ ]:
GZIP_PATH = OUT_DIR / "synthetic_logs.jsonl.gz"

with RAW_PATH.open("rb") as f_in, gzip.open(GZIP_PATH, "wb", compresslevel=9) as f_out:
    shutil.copyfileobj(f_in, f_out)

print("gzip size MB:", file_size(GZIP_PATH) / 1e6)

gzip size MB: 1.581074


In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zstandard"], check=True)
import zstandard as zstd

def zstd_compress(input_path: Path, output_path: Path, level: int):
    cctx = zstd.ZstdCompressor(level=level)
    with input_path.open("rb") as f_in, output_path.open("wb") as f_out:
        cctx.copy_stream(f_in, f_out)

ZSTD3_PATH = OUT_DIR / "synthetic_logs.jsonl.zst3"
ZSTD19_PATH = OUT_DIR / "synthetic_logs.jsonl.zst19"

zstd_compress(RAW_PATH, ZSTD3_PATH, level=3)
zstd_compress(RAW_PATH, ZSTD19_PATH, level=19)

print("zstd -3 size MB:", file_size(ZSTD3_PATH) / 1e6)
print("zstd -19 size MB:", file_size(ZSTD19_PATH) / 1e6)

zstd -3 size MB: 2.566277
zstd -19 size MB: 1.240982


In [ ]:
CLP_IMAGE = "ghcr.io/y-scope/clp/clp-core-x86-ubuntu-jammy:main"

run(["docker", "pull", CLP_IMAGE])

+ docker pull ghcr.io/y-scope/clp/clp-core-x86-ubuntu-jammy:main


FileNotFoundError: [Errno 2] No such file or directory: 'docker'

In [ ]:
inspect_cmd = r"""
set -e
echo "PWD=$(pwd)"
echo "Listing likely dirs:"
ls -la
echo
echo "Finding clp-s:"
find / -name clp-s -type f 2>/dev/null | head -20
"""

run([
    "docker", "run", "--rm",
    CLP_IMAGE,
    "/bin/bash", "-lc", inspect_cmd
])

In [ ]:
CLP_S = "./clp-s"

In [ ]:
CLP_S = "./clp-s"  # change if Cell 7 found a different absolute path

# Clean previous CLP output.
if CLP_OUT_DIR.exists():
    shutil.rmtree(CLP_OUT_DIR)
CLP_OUT_DIR.mkdir(parents=True, exist_ok=True)

uid = os.getuid()
gid = os.getgid()

compress_cmd = f"""
set -e
mkdir -p /mnt/data/archives
{CLP_S} c \
  --timestamp-key timestamp \
  --target-encoded-size 1073741824 \
  --compression-level 6 \
  /mnt/data/archives \
  /mnt/logs/{RAW_PATH.name}

echo "Archive files:"
find /mnt/data/archives -type f -maxdepth 3 -print
echo "Archive total bytes:"
du -sb /mnt/data/archives || du -sk /mnt/data/archives
"""

run([
    "docker", "run", "--rm",
    "-u", f"{uid}:{gid}",
    "--volume", f"{DATA_DIR}:/mnt/logs",
    "--volume", f"{CLP_OUT_DIR}:/mnt/data",
    CLP_IMAGE,
    "/bin/bash", "-lc", compress_cmd
])

In [ ]:
clp_archive_dir = CLP_OUT_DIR / "archives"

sizes = {
    "raw JSONL": file_size(RAW_PATH),
    "gzip -9": file_size(GZIP_PATH),
    "zstd -3": file_size(ZSTD3_PATH),
    "zstd -19": file_size(ZSTD19_PATH),
    "CLP / μSlope": dir_size(clp_archive_dir),
}

df = pd.DataFrame([
    {
        "method": method,
        "size_bytes": size,
        "size_MB": size / 1e6,
        "compression_ratio": sizes["raw JSONL"] / size,
    }
    for method, size in sizes.items()
])

df

In [ ]:
plot_df = df[df["method"] != "raw JSONL"].copy()

plt.figure(figsize=(8, 4))
plt.bar(plot_df["method"], plot_df["compression_ratio"])
plt.ylabel("Compression ratio = raw JSONL size / compressed size")
plt.xlabel("Method")
plt.title("Reduced μSlope / CLP Compression-Ratio Reproduction")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
decomp_cmd = f"""
set -e
rm -rf /mnt/data/decompressed
mkdir -p /mnt/data/decompressed
{CLP_S} x /mnt/data/archives /mnt/data/decompressed

echo "Decompressed files:"
find /mnt/data/decompressed -type f -print
"""

run([
    "docker", "run", "--rm",
    "-u", f"{uid}:{gid}",
    "--volume", f"{DATA_DIR}:/mnt/logs",
    "--volume", f"{CLP_OUT_DIR}:/mnt/data",
    CLP_IMAGE,
    "/bin/bash", "-lc", decomp_cmd
])